In [2]:
#!/usr/bin/env python3
# =========================================================
# 7-STAGE ENSEMBLE LEGAL SUMMARIZATION PIPELINE  (v3 — utf8 fix)
# =========================================================
# KEY FIX: All open() calls now use encoding='utf-8'
#           so Unicode characters (smart quotes, etc.) in
#           generated summaries are saved without error.
#
# BEHAVIOUR: Stages 1 & 2 annotation load from cached JSON
#            files written in the previous run, so the script
#            resumes from Stage 3 onward.
# =========================================================

import os, json, re, torch, torch.nn as nn, torch.nn.functional as F
import numpy as np
from tqdm import tqdm
from collections import Counter, defaultdict
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.metrics.pairwise import cosine_similarity
from transformers import (
    AutoTokenizer, AutoModel, AutoModelForSeq2SeqLM,
    LEDForConditionalGeneration, PegasusForConditionalGeneration
)
import evaluate, nltk

# ── NLTK ──────────────────────────────────────────────────
print("📥 Downloading NLTK data...")
for res in ['tokenizers/punkt', 'tokenizers/punkt_tab']:
    try:
        nltk.data.find(res)
    except LookupError:
        nltk.download(res.split('/')[-1], quiet=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Device: {device}")

# =========================================================
# SPACY LEGAL NER
# =========================================================
import spacy

print("\n📚 Loading spaCy Legal NER model...")
_NLP        = None
_NLP_SOURCE = None

def _load_spacy_nlp():
    global _NLP, _NLP_SOURCE
    for model_name in ["en_legal_ner_trf", "en_core_web_trf", "en_core_web_sm"]:
        try:
            _NLP        = spacy.load(model_name)
            _NLP_SOURCE = model_name
            has_boundary = any(
                pipe in _NLP.pipe_names
                for pipe in ("parser", "senter", "sentencizer")
            )
            if not has_boundary:
                _NLP.add_pipe("sentencizer")
                print(f"  ℹ️  Added 'sentencizer' to '{model_name}' pipeline")
            print(f"  ✅ Loaded spaCy model: {model_name}  (pipes: {_NLP.pipe_names})")
            return
        except OSError:
            print(f"  ⚠️  spaCy model '{model_name}' not found, trying next…")
    _NLP        = None
    _NLP_SOURCE = "none"
    print("  ❌ No spaCy model available — entity extraction will be skipped.")

_load_spacy_nlp()

_SPACY_LABEL_TO_CATEGORY = {
    "COURT":        "courts",   "PETITIONER":   "parties",
    "RESPONDENT":   "parties",  "JUDGE":        "judges",
    "LAWYER":       "lawyers",  "STATUTE":      "statutes",
    "PROVISION":    "provisions","PRECEDENT":   "cases",
    "CASE_NUMBER":  "cases",    "ORG":          "organizations",
    "GPE":          "locations","DATE":         "dates",
    "WITNESS":      "persons",  "OTHER_PERSON": "persons",
    "PERSON":       "persons",  "NORP":         "organizations",
    "FAC":          "locations","LOC":          "locations",
    "PRODUCT":      "organizations","EVENT":    "events",
    "LAW":          "statutes", "CARDINAL":     "numbers",
    "ORDINAL":      "numbers",  "PERCENT":      "numbers",
    "MONEY":        "numbers",  "QUANTITY":     "numbers",
    "TIME":         "dates",
}

_LEGAL_CRITICAL_CATEGORIES = {
    "cases", "statutes", "provisions", "courts", "judges",
    "parties", "lawyers", "organizations"
}


def extract_entities_spacy(text):
    entities = defaultdict(set)
    if _NLP is None:
        return {cat: [] for cat in _LEGAL_CRITICAL_CATEGORIES}
    sentences_for_ner = sent_tokenize(text)
    CHUNK_SIZE = 20
    for i in range(0, len(sentences_for_ner), CHUNK_SIZE):
        chunk = " ".join(sentences_for_ner[i : i + CHUNK_SIZE])
        try:
            doc = _NLP(chunk[:5000])
        except Exception as e:
            print(f"    [NER] spaCy error on chunk {i}: {e}")
            continue
        for ent in doc.ents:
            label    = ent.label_.upper().strip()
            ent_text = ent.text.strip()
            if len(ent_text) < 3 or ent_text.isdigit():
                continue
            category = _SPACY_LABEL_TO_CATEGORY.get(label)
            if category:
                entities[category].add(ent_text)
    result = {cat: sorted(list(ents)) for cat, ents in entities.items()}
    for cat in _LEGAL_CRITICAL_CATEGORIES:
        if cat not in result:
            result[cat] = []
    return result


def extract_relations_spacy(text, entities_dict):
    """Sentence-by-sentence dep-parse — no doc.sents needed (E030 fix)."""
    relations = []
    if _NLP is None:
        return relations

    LEGAL_VERBS = {
        "apply": "applies",   "applies": "applies",   "applied": "applies",
        "overrule": "overrules","overrules":"overrules","overruled":"overrules",
        "cite": "cites",      "cites": "cites",       "cited": "cites",
        "dismiss": "dismisses","dismisses":"dismisses","dismissed":"dismisses",
        "uphold": "upholds",  "upholds": "upholds",   "upheld": "upholds",
        "allow": "allows",    "allows": "allows",     "allowed": "allows",
        "follow": "follows",  "follows": "follows",   "followed": "follows",
        "distinguish":"distinguishes","distinguishes":"distinguishes","distinguished":"distinguishes",
        "rely": "relies_on",  "relies": "relies_on",  "relied": "relies_on",
        "refer": "referred_to","refers":"referred_to","referred":"referred_to",
    }

    def _find_entity(span, ents_lookup):
        if span in ents_lookup:
            return ents_lookup[span]
        for key, val in ents_lookup.items():
            if key in span or span in key:
                return val
        return None

    for sentence_text in sent_tokenize(text):
        try:
            doc = _NLP(sentence_text[:1000])
        except Exception:
            continue
        chunk_ents = {}
        for ent in doc.ents:
            cat = _SPACY_LABEL_TO_CATEGORY.get(ent.label_.upper())
            if cat and cat in _LEGAL_CRITICAL_CATEGORIES:
                chunk_ents[ent.text.lower()] = ent.text
        root = None
        for token in doc:
            if token.dep_ == "ROOT" and token.pos_ == "VERB":
                root = token
                break
        if root is None:
            continue
        rel_type = LEGAL_VERBS.get(root.lemma_.lower())
        if rel_type is None:
            continue
        subjects, objects = [], []
        for child in root.children:
            if child.dep_ in ("nsubj", "nsubjpass"):
                subjects.append(" ".join(t.text for t in child.subtree).lower().strip())
            if child.dep_ in ("dobj", "pobj", "attr", "oprd", "nmod"):
                objects.append(" ".join(t.text for t in child.subtree).lower().strip())
        for subj_span in subjects:
            subj_ent = _find_entity(subj_span, chunk_ents)
            if not subj_ent:
                continue
            for obj_span in objects:
                obj_ent = _find_entity(obj_span, chunk_ents)
                if obj_ent and subj_ent != obj_ent:
                    relations.append((subj_ent, rel_type, obj_ent))

    seen = set()
    unique = []
    for r in relations:
        key = (r[0].lower(), r[1], r[2].lower())
        if key not in seen:
            seen.add(key)
            unique.append(r)
    return unique


# =========================================================
# GLOBAL CONFIGURATION
# =========================================================

MODEL_PATHS = {
    "BART":       "BART",
    "LED":        "LED",
    "PEGASUS":    "PEGASUS",
    "ROLE_MODEL": "best_RR_model"
}

TRAIN_JSON_PATH = "train.json"
TEST_JSON_PATH  = "test.json"
OUTPUT_DIR      = "./pipeline_7stage_v3_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_CONFIG = {
    "BART":    {"max_input": 1024, "max_output": 256, "num_beams": 4},
    "LED":     {"max_input": 4096, "max_output": 512, "num_beams": 4},
    "PEGASUS": {"max_input": 1024, "max_output": 256, "num_beams": 4}
}

HSLN_LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER", "ARG_RESPONDENT",
    "ANALYSIS", "STA", "PRE_RELIED", "PRE_NOT_RELIED", "RATIO", "RPC", "NONE"
]
NUM_HSLN_LABELS = 13

LABELS_8 = [
    "PREAMBLE", "FACTS", "ISSUE", "ARGUMENTS",
    "REASONING", "PRECEDENT_RELIED", "PRECEDENT_NOT_RELIED", "PROCEDURAL"
]

LABEL_MAPPING_13_TO_8 = {
    "PREAMBLE": "PREAMBLE", "ISSUE": "ISSUE",
    "FAC": "FACTS", "PRE_RELIED": "PRECEDENT_RELIED",
    "PRE_NOT_RELIED": "PRECEDENT_NOT_RELIED",
    "ARG_PETITIONER": "ARGUMENTS", "ARG_RESPONDENT": "ARGUMENTS",
    "ANALYSIS": "REASONING", "RATIO": "REASONING", "RPC": "REASONING",
    "STA": "PROCEDURAL", "RLC": "PROCEDURAL", "NONE": "PROCEDURAL"
}

def map_13_to_8(labels_13):
    return [LABEL_MAPPING_13_TO_8.get(l, "PROCEDURAL") for l in labels_13]

HYBRID_ALPHA = 0.6
HYBRID_BETA  = 0.4

COMPRESSION_RATIOS = {"BART": 0.12, "PEGASUS": 0.12, "LED": 0.40}
MIN_SENTENCES      = {"BART": 30,   "PEGASUS": 30,   "LED": 100}
MAX_SENTENCES      = {"BART": 150,  "PEGASUS": 150,  "LED": 400}
MAX_TRAIN_SAMPLES  = 3000

KG_CRITICAL_ROLES     = ["ISSUE", "REASONING", "PRECEDENT_RELIED"]
RETRIEVAL_TOP_K       = 10
GRAPH_TRAVERSAL_DEPTH = 3
PATH_RANKING_ALPHA    = 0.4
PATH_RANKING_BETA     = 0.3
PATH_RANKING_GAMMA    = 0.3
FINAL_TOP_K_PATHS     = 5

STRUCT_TAGS = {
    "FACTS":            "[FACTS]",
    "PRECEDENT_RELIED": "[PRECEDENT]",
    "PROCEDURAL":       "[STATUTE]",
    "REASONING":        "[RATIO]",
    "ISSUE":            "[DECISION]"
}

RL_REWARD_WEIGHTS_DEFAULT = {
    "factual_faithfulness":   0.30,
    "citation_correctness":   0.20,
    "reasoning_completeness": 0.25,
    "rhetorical_balance":     0.15,
    "abstraction_quality":    0.10
}
RL_CASE_TYPE_WEIGHTS = {
    "constitutional": {
        "factual_faithfulness":   0.25, "citation_correctness":   0.15,
        "reasoning_completeness": 0.35, "rhetorical_balance":     0.15,
        "abstraction_quality":    0.10
    },
    "criminal": {
        "factual_faithfulness":   0.40, "citation_correctness":   0.20,
        "reasoning_completeness": 0.20, "rhetorical_balance":     0.10,
        "abstraction_quality":    0.10
    },
    "civil": {
        "factual_faithfulness":   0.30, "citation_correctness":   0.25,
        "reasoning_completeness": 0.20, "rhetorical_balance":     0.15,
        "abstraction_quality":    0.10
    }
}
RL_KL_PENALTY_COEF = 0.02
RL_CLIP_EPSILON    = 0.2

print("\n" + "="*70)
print("7-STAGE PIPELINE  (v3 — utf8 fix)")
print(f"  NER backend: {_NLP_SOURCE}")
print("  OUTPUT_DIR:", OUTPUT_DIR)
print("="*70)

# =========================================================
# HSLN MODEL
# =========================================================

class PrototypeAttention(nn.Module):
    def __init__(self, dim, heads=8, dropout=0.1):
        super().__init__()
        self.h = heads; self.dh = dim // heads
        self.q = nn.Linear(dim, dim, bias=False)
        self.k = nn.Linear(dim, dim, bias=False)
        self.v = nn.Linear(dim, dim, bias=False)
        self.o = nn.Linear(dim, dim, bias=False)
        self.ln = nn.LayerNorm(dim)
        self.drop = nn.Dropout(dropout)
    def forward(self, x, p):
        B, T, D = x.shape
        Q = self.q(x).view(B, T, self.h, self.dh)
        K = self.k(p).view(p.size(0), self.h, self.dh)
        V = self.v(p).view(p.size(0), self.h, self.dh)
        Q = F.normalize(Q, dim=-1); K = F.normalize(K, dim=-1)
        outs = []
        for h in range(self.h):
            a = (Q[:, :, h] @ K[:, h].T).softmax(-1)
            outs.append(self.drop(a) @ V[:, h])
        return self.ln(x + self.drop(self.o(torch.cat(outs, -1))))

class RPL(nn.Module):
    def __init__(self, dim, num_labels):
        super().__init__()
        self.proto = nn.Parameter(torch.randn(num_labels, dim))
    def forward(self, h):
        return F.normalize(h, dim=-1) @ F.normalize(self.proto, dim=-1).T

class RTM(nn.Module):
    def __init__(self, num_labels, lam):
        super().__init__()
        self.A = nn.Parameter(torch.zeros(num_labels, num_labels))
        self.lam = lam
    def forward(self, logits):
        lp = logits.log_softmax(-1)
        for t in range(1, logits.size(1)):
            tr = torch.logsumexp(lp[:, t-1].unsqueeze(1) + self.A.log_softmax(-1), -1)
            logits[:, t] += self.lam * tr
        return logits

class HSLNModel(nn.Module):
    def __init__(self, embedding_dim=768, lstm_hidden=384, num_labels=13,
                 dropout=0.3, rtm_lambda=0.05):
        super().__init__()
        self.att  = PrototypeAttention(embedding_dim, dropout=dropout)
        self.lstm = nn.LSTM(embedding_dim, lstm_hidden, 2,
                            bidirectional=True, batch_first=True, dropout=dropout)
        self.cls  = nn.Linear(lstm_hidden * 2, num_labels)
        self.rpl  = RPL(lstm_hidden * 2, num_labels)
        self.alpha = nn.Parameter(torch.tensor(2.0))
        self.rtm  = RTM(num_labels, rtm_lambda)
        self.register_buffer('prototypes', torch.randn(num_labels, embedding_dim))
    def forward(self, embeddings, lengths=None):
        x = self.att(embeddings, self.prototypes)
        if lengths is not None:
            pck = nn.utils.rnn.pack_padded_sequence(
                x, lengths.cpu(), batch_first=True, enforce_sorted=False)
            o, _ = self.lstm(pck)
            o, _ = nn.utils.rnn.pad_packed_sequence(o, batch_first=True)
        else:
            o, _ = self.lstm(x)
        a = torch.sigmoid(self.alpha)
        return self.rtm(a * self.cls(o) + (1 - a) * self.rpl(o))
    def predict(self, emb, lengths=None):
        return torch.argmax(self.forward(emb, lengths), dim=-1)

# =========================================================
# LOAD MODELS
# =========================================================

print("\n📚 Loading HSLN role classifier...")
_cfg = {}
cfg_path = os.path.join(MODEL_PATHS["ROLE_MODEL"], "config.json")
if os.path.exists(cfg_path):
    with open(cfg_path, encoding='utf-8') as f:       # utf-8 FIX
        _cfg = json.load(f)

role_model = HSLNModel(
    embedding_dim=_cfg.get('embedding_dim', 768),
    lstm_hidden=_cfg.get('lstm_hidden', 384),
    num_labels=NUM_HSLN_LABELS,
    dropout=_cfg.get('dropout', 0.3),
    rtm_lambda=_cfg.get('rtm_lambda', 0.05)
)
sd = torch.load(os.path.join(MODEL_PATHS["ROLE_MODEL"], "pytorch_model.bin"),
                map_location=device)
sd.pop('prototypes', None)
role_model.load_state_dict(sd, strict=False)
role_model.prototypes = torch.load(
    os.path.join(MODEL_PATHS["ROLE_MODEL"], "prototypes.pt"), map_location=device)
role_model.to(device).eval()
print("✅ HSLN loaded")

print("\n📚 Loading InLegalBERT...")
legal_tokenizer = AutoTokenizer.from_pretrained("law-ai/InLegalBERT")
legal_model     = AutoModel.from_pretrained("law-ai/InLegalBERT").to(device)
legal_model.eval()
print("✅ InLegalBERT loaded")

print("\n📚 Loading summarization models...")
models = {}; tokenizers = {}
for name, cls, path in [
    ("BART",    AutoModelForSeq2SeqLM,          MODEL_PATHS["BART"]),
    ("LED",     LEDForConditionalGeneration,     MODEL_PATHS["LED"]),
    ("PEGASUS", PegasusForConditionalGeneration, MODEL_PATHS["PEGASUS"])
]:
    print(f"  Loading {name}...")
    tokenizers[name] = AutoTokenizer.from_pretrained(path)
    models[name]     = cls.from_pretrained(path).to(device)
    models[name].eval()
    print(f"  ✅ {name} loaded")

print("\n📊 Loading evaluation metrics...")
rouge_metric     = evaluate.load("rouge")
bertscore_metric = evaluate.load("bertscore")
print("✅ Metrics loaded\n")

# =========================================================
# SHARED UTILITIES
# =========================================================

@torch.no_grad()
def embed_texts(texts, batch_size=16):
    if not texts:
        return np.array([])
    embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc   = legal_tokenizer(batch, padding=True, truncation=True,
                                max_length=512, return_tensors="pt").to(device)
        out   = legal_model(**enc).last_hidden_state
        mask  = enc["attention_mask"].unsqueeze(-1).float()
        embs.append(((out * mask).sum(1) / mask.sum(1)).cpu().numpy())
    return np.vstack(embs)

@torch.no_grad()
def classify_roles(sentences, batch_size=16):
    if not sentences:
        return []
    preds13 = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        inp   = legal_tokenizer(batch, padding=True, truncation=True,
                                max_length=128, return_tensors="pt").to(device)
        emb   = legal_model(**inp).last_hidden_state.mean(dim=1).unsqueeze(1)
        lens  = torch.ones(len(batch), dtype=torch.long).to(device)
        preds13.extend(role_model.predict(emb, lens)[:, 0].cpu().tolist())
    return map_13_to_8([HSLN_LABELS[p] for p in preds13])

def get_adaptive_targets(doc_len):
    out = {}
    for m, r in COMPRESSION_RATIOS.items():
        t = int(doc_len * r)
        out[m] = max(MIN_SENTENCES[m], min(MAX_SENTENCES[m], t))
    return out

def generate_summary(model_name, text):
    tok = tokenizers[model_name]
    cfg = MODEL_CONFIG[model_name]
    inp = tok(text, return_tensors="pt", truncation=True,
              max_length=cfg["max_input"]).to(device)
    with torch.no_grad():
        if model_name == "LED":
            gam = torch.zeros_like(inp["attention_mask"])
            gam[:, 0] = 1
            ids = models[model_name].generate(
                inp["input_ids"], attention_mask=inp["attention_mask"],
                global_attention_mask=gam,
                max_length=cfg["max_output"], num_beams=cfg["num_beams"],
                early_stopping=True, no_repeat_ngram_size=3)
        else:
            ids = models[model_name].generate(
                inp["input_ids"], attention_mask=inp["attention_mask"],
                max_length=cfg["max_output"], num_beams=cfg["num_beams"],
                early_stopping=True, no_repeat_ngram_size=3)
    return tok.decode(ids[0], skip_special_tokens=True)

def compute_metrics(predictions, references):
    rs = rouge_metric.compute(predictions=predictions, references=references,
                               use_stemmer=True)
    bs = bertscore_metric.compute(predictions=predictions, references=references,
                                  model_type="roberta-large", lang="en",
                                  device=device, batch_size=8)
    return {
        "rouge1":       float(rs["rouge1"]),
        "rouge2":       float(rs["rouge2"]),
        "rougeL":       float(rs["rougeL"]),
        "bertscore_f1": float(np.mean(bs["f1"]))
    }

def detect_case_type(text):
    text_lower = text.lower()
    if any(k in text_lower for k in ["constitution", "fundamental right", "article 21"]):
        return "constitutional"
    if any(k in text_lower for k in ["ipc", "criminal", "accused", "fir", "murder", "theft"]):
        return "criminal"
    return "civil"

# =========================================================
# STAGE 1: ROLE LEARNING
# =========================================================

def stage1_role_learning(train_data):
    print("\n" + "="*70)
    print("STAGE 1: ROLE LEARNING (TRAINING PHASE)")
    print("="*70)

    ann_path  = os.path.join(OUTPUT_DIR, "train_annotations_8label.json")
    cont_path = os.path.join(OUTPUT_DIR, "role_contribution_scores.json")
    wts_path  = os.path.join(OUTPUT_DIR, "normalized_role_weights.json")

    if os.path.exists(ann_path):
        print(f"📂 Loading cached train annotations from {ann_path}")
        with open(ann_path, encoding='utf-8') as f:    # utf-8 FIX
            train_annotations = json.load(f)
    else:
        train_annotations = []
        for idx, item in enumerate(tqdm(train_data, desc="  Annotating")):
            judgment  = item.get("judgment", "")
            doc_id    = item.get("id", idx)
            sentences = sent_tokenize(judgment)
            if not sentences:
                continue
            roles = classify_roles(sentences)
            ann   = {"doc_id": doc_id, "total_sentences": len(sentences), "sentences": []}
            for i, (s, r) in enumerate(zip(sentences, roles)):
                ann["sentences"].append({"index": i, "sentence": s, "role": r})
            train_annotations.append(ann)
        with open(ann_path, 'w', encoding='utf-8') as f:  # utf-8 FIX
            json.dump(train_annotations, f, indent=2, ensure_ascii=False)
        print(f"  ✅ Saved: {ann_path}")

    if os.path.exists(cont_path):
        print(f"📂 Loading cached role contribution scores from {cont_path}")
        with open(cont_path, encoding='utf-8') as f:  # utf-8 FIX
            cont_data = json.load(f)
        role_scores = cont_data["role_scores"]
    else:
        role_total = Counter(); role_summary = Counter()
        for ann, item in tqdm(zip(train_annotations, train_data),
                              total=len(train_annotations), desc="  Contribution"):
            ref_sents  = sent_tokenize(item.get("reference_summary", ""))
            doc_sents  = [s["sentence"] for s in ann["sentences"]]
            doc_roles  = [s["role"]     for s in ann["sentences"]]
            for r in doc_roles:
                role_total[r] += 1
            if doc_sents and ref_sents:
                doc_emb = embed_texts(doc_sents)
                ref_emb = embed_texts(ref_sents)
                for re_emb in ref_emb:
                    sims = cosine_similarity([re_emb], doc_emb)[0]
                    best = int(np.argmax(sims))
                    if sims[best] > 0.7:
                        role_summary[doc_roles[best]] += 1
        role_scores = {r: role_summary[r] / role_total[r] if role_total[r] > 0 else 0.0
                       for r in LABELS_8}
        cont = {"role_scores": role_scores,
                "role_total_counts":   dict(role_total),
                "role_summary_counts": dict(role_summary)}
        with open(cont_path, 'w', encoding='utf-8') as f:  # utf-8 FIX
            json.dump(cont, f, indent=2, ensure_ascii=False)
        print(f"  ✅ Saved: {cont_path}")

    if os.path.exists(wts_path):
        print(f"📂 Loading cached normalized weights from {wts_path}")
        with open(wts_path, encoding='utf-8') as f:   # utf-8 FIX
            normalized_weights = json.load(f)["normalized_weights"]
    else:
        total = sum(role_scores.values())
        normalized_weights = {r: s / total if total > 0 else 1.0/len(LABELS_8)
                              for r, s in role_scores.items()}
        with open(wts_path, 'w', encoding='utf-8') as f:  # utf-8 FIX
            json.dump({"normalized_weights": normalized_weights,
                       "original_scores":    role_scores}, f, indent=2)
        print(f"  ✅ Saved: {wts_path}")

    print("\n  📊 Learned Role Importance Weights:")
    print("  " + "-"*60)
    for r, w in sorted(normalized_weights.items(), key=lambda x: x[1], reverse=True):
        bar = "█" * int(w * 100)
        print(f"  {r:<25}: {w:.4f}  {bar}")
    print("  " + "-"*60)
    return train_annotations, normalized_weights

# =========================================================
# STAGE 2
# =========================================================

def stage2_hybrid_selection(doc_annotation, normalized_weights, target_sentences):
    sentences  = [s["sentence"] for s in doc_annotation["sentences"]]
    roles      = [s["role"]     for s in doc_annotation["sentences"]]
    if not sentences:
        return "", {}
    embeddings = embed_texts(sentences)
    centroid   = embeddings.mean(axis=0, keepdims=True)
    salience   = cosine_similarity(embeddings, centroid).squeeze()
    scored = []
    for idx, (role, sal) in enumerate(zip(roles, salience)):
        rw    = normalized_weights.get(role, 0.0)
        score = HYBRID_ALPHA * rw * 10 + HYBRID_BETA * float(sal)
        scored.append({"index": idx, "score": score, "role": role,
                       "role_weight": rw, "salience": float(sal),
                       "sentence": sentences[idx]})
    ranked   = sorted(scored, key=lambda x: x["score"], reverse=True)
    selected = ranked[:target_sentences]
    indices  = sorted([s["index"] for s in selected])
    filtered_text = " ".join(sentences[i] for i in indices)
    return filtered_text, {
        "total_available":   len(sentences),
        "target":            target_sentences,
        "selected":          len(indices),
        "role_distribution": dict(Counter(s["role"] for s in selected))
    }

def annotate_test_documents(test_data):
    ann_path = os.path.join(OUTPUT_DIR, "test_annotations_8label.json")
    if os.path.exists(ann_path):
        print(f"📂 Loading cached test annotations from {ann_path}")
        with open(ann_path, encoding='utf-8') as f:   # utf-8 FIX
            return json.load(f)
    print("  Annotating test documents...")
    annotations = []
    for idx, item in enumerate(tqdm(test_data, desc="  Test annotation")):
        judgment  = item.get("judgment", "")
        doc_id    = item.get("id", idx)
        sentences = sent_tokenize(judgment)
        if not sentences:
            continue
        roles = classify_roles(sentences)
        ann   = {"doc_id": doc_id, "total_sentences": len(sentences), "sentences": []}
        for i, (s, r) in enumerate(zip(sentences, roles)):
            ann["sentences"].append({"index": i, "sentence": s, "role": r})
        annotations.append(ann)
    with open(ann_path, 'w', encoding='utf-8') as f:  # utf-8 FIX
        json.dump(annotations, f, indent=2, ensure_ascii=False)
    print(f"  ✅ Saved: {ann_path}")
    return annotations

# =========================================================
# STAGE 3: KG
# =========================================================

def build_role_conditioned_kg(filtered_text, doc_annotation):
    print(f"    [Stage 3] Building KG  (NER: {_NLP_SOURCE})")
    role_chunks = defaultdict(list)
    for s in doc_annotation["sentences"]:
        role_chunks[s["role"]].append(s["sentence"])

    chunk_embeddings = {}
    for role, sents in role_chunks.items():
        if sents:
            chunk_embeddings[role] = embed_texts([" ".join(sents[:5])])[0]

    entities = extract_entities_spacy(filtered_text)
    ent_summary = {cat: len(lst) for cat, lst in entities.items() if lst}
    print(f"    [Stage 3] Entities: {ent_summary}")

    relations = extract_relations_spacy(filtered_text, entities)
    print(f"    [Stage 3] Relations: {len(relations)}")

    nodes = {}; node_id = 0
    entity_text_to_nid = {}
    for category, ent_list in entities.items():
        if category not in _LEGAL_CRITICAL_CATEGORIES:
            continue
        for ent_text in ent_list:
            nid = f"ENT_{node_id}"
            nodes[nid] = {"type": "entity", "entity_type": category,
                          "text": ent_text, "degree": 0,
                          "entity_centrality": 0.0, "path_centrality": 0.0}
            entity_text_to_nid[(category, ent_text.lower())] = nid
            node_id += 1

    chunk_nodes = {}
    for role, sents in role_chunks.items():
        nid = f"CHUNK_{role}"
        nodes[nid] = {"type": "role_chunk", "role": role,
                      "text": " ".join(sents[:3]), "sentence_count": len(sents),
                      "degree": 0,
                      "embedding": chunk_embeddings.get(role, np.zeros(768)).tolist(),
                      "entity_centrality": 0.0, "path_centrality": 0.0}
        chunk_nodes[role] = nid

    edges = []

    def _find_entity_nid(text):
        text_lower = text.lower()
        for (cat, ent_lower), nid in entity_text_to_nid.items():
            if ent_lower == text_lower:
                return nid
        for (cat, ent_lower), nid in entity_text_to_nid.items():
            if text_lower[:15] in ent_lower or ent_lower[:15] in text_lower:
                return nid
        return None

    for subj_text, rel_type, obj_text in relations:
        src_nid = _find_entity_nid(subj_text)
        tgt_nid = _find_entity_nid(obj_text)
        if src_nid and tgt_nid and src_nid != tgt_nid:
            edges.append({"src": src_nid, "tgt": tgt_nid,
                          "type": "legal_relation", "relation": rel_type,
                          "subj": subj_text, "obj": obj_text})
            nodes[src_nid]["degree"] += 1
            nodes[tgt_nid]["degree"] += 1

    for category, ent_list in entities.items():
        for ent_text in ent_list:
            src_nid = _find_entity_nid(ent_text)
            if not src_nid:
                continue
            ent_lower = ent_text.lower()
            for role, sents in role_chunks.items():
                chunk_nid = chunk_nodes.get(role)
                if not chunk_nid:
                    continue
                if any(ent_lower[:10] in s.lower() for s in sents[:5]):
                    edges.append({"src": src_nid, "tgt": chunk_nid,
                                  "type": "entity_chunk_link", "relation": "mentioned_in"})
                    nodes[src_nid]["degree"]   += 1
                    nodes[chunk_nid]["degree"] += 1

    for src_role, tgt_role, rel in [
        ("ISSUE", "FACTS", "precedes"), ("FACTS", "ARGUMENTS", "precedes"),
        ("ARGUMENTS", "REASONING", "leads_to"),
        ("REASONING", "PRECEDENT_RELIED", "supported_by"),
        ("PRECEDENT_RELIED", "REASONING", "informs"),
        ("REASONING", "ISSUE", "resolves")
    ]:
        if src_role in chunk_nodes and tgt_role in chunk_nodes:
            src_nid = chunk_nodes[src_role]; tgt_nid = chunk_nodes[tgt_role]
            edges.append({"src": src_nid, "tgt": tgt_nid,
                          "type": "rhetorical_transition", "relation": rel})
            nodes[src_nid]["degree"] += 1
            nodes[tgt_nid]["degree"] += 1

    max_degree = max((n["degree"] for n in nodes.values()), default=1)
    for nid in nodes:
        nodes[nid]["entity_centrality"] = nodes[nid]["degree"] / max(max_degree, 1)

    path_counts = Counter()
    for e in edges:
        path_counts[e["src"]] += 1; path_counts[e["tgt"]] += 1
    max_path = max(path_counts.values(), default=1)
    for nid in nodes:
        nodes[nid]["path_centrality"] = path_counts.get(nid, 0) / max(max_path, 1)

    role_coverage = len(chunk_nodes) / len(LABELS_8)

    def dfs_depth(start, visited=None):
        if visited is None: visited = set()
        if start in visited: return 0
        visited.add(start)
        children = [e["tgt"] for e in edges
                    if e["src"] == start and e["type"] == "rhetorical_transition"]
        return 1 + max((dfs_depth(c, visited.copy()) for c in children), default=0)

    reasoning_depth = max((dfs_depth(nid) for nid in chunk_nodes.values()), default=0)

    critical_path_sents = []
    for role in ["PRECEDENT_RELIED", "PROCEDURAL", "REASONING", "ISSUE"]:
        role_sents = [s["sentence"] for s in doc_annotation["sentences"]
                      if s["role"] == role]
        if role_sents:
            critical_path_sents.append(role_sents[0])

    kg = {
        "nodes": nodes, "edges": edges, "entities": entities,
        "chunk_nodes": chunk_nodes,
        "chunk_embeddings": {r: e.tolist() for r, e in chunk_embeddings.items()},
        "ner_backend": _NLP_SOURCE,
        "graph_metrics": {
            "role_coverage":      round(role_coverage, 4),
            "reasoning_depth":    reasoning_depth,
            "total_nodes":        len(nodes),
            "total_edges":        len(edges),
            "legal_relations":    sum(1 for e in edges if e["type"] == "legal_relation"),
            "entity_chunk_links": sum(1 for e in edges if e["type"] == "entity_chunk_link"),
            "rhet_transitions":   sum(1 for e in edges if e["type"] == "rhetorical_transition"),
            "entities_by_cat":    {cat: len(lst) for cat, lst in entities.items() if lst}
        },
        "critical_path_sentences": critical_path_sents
    }
    print(f"    [Stage 3] KG: {len(nodes)} nodes, {len(edges)} edges, "
          f"coverage={role_coverage:.2f}, depth={reasoning_depth}")
    return kg

# =========================================================
# STAGE 4: HYBRID RETRIEVAL
# =========================================================

class HybridRetriever:
    def __init__(self, kg, doc_annotation, normalized_weights):
        self.kg                 = kg
        self.doc_annotation     = doc_annotation
        self.normalized_weights = normalized_weights
        self.sentences          = [s["sentence"] for s in doc_annotation["sentences"]]
        self.roles              = [s["role"]     for s in doc_annotation["sentences"]]
        self.sent_embeddings    = (embed_texts(self.sentences)
                                   if self.sentences else np.zeros((0, 768)))

    def vector_retrieval(self, query_text, top_k=RETRIEVAL_TOP_K):
        if len(self.sentences) == 0:
            return []
        sims       = cosine_similarity(embed_texts([query_text]), self.sent_embeddings)[0]
        ranked_idx = np.argsort(sims)[::-1][:top_k]
        return [{"sentence": self.sentences[i], "role": self.roles[i],
                 "sim_score": float(sims[i]), "sent_idx": int(i)}
                for i in ranked_idx]

    def graph_traversal(self, start_roles=None, depth=GRAPH_TRAVERSAL_DEPTH):
        if start_roles is None:
            start_roles = ["PRECEDENT_RELIED", "ISSUE"]
        chunk_nodes = self.kg["chunk_nodes"]
        edges       = self.kg["edges"]
        adj         = defaultdict(list)
        for e in edges:
            if e["type"] == "rhetorical_transition":
                adj[e["src"]].append((e["tgt"], e["relation"]))
        all_paths = []

        def dfs(nid, path_sents, path_roles, visited, hop):
            node       = self.kg["nodes"].get(nid, {})
            role       = node.get("role", "")
            role_sents = [s["sentence"] for s in self.doc_annotation["sentences"]
                          if s["role"] == role][:2]
            new_sents  = path_sents + role_sents
            new_roles  = path_roles + [role]
            if hop >= depth or not adj[nid]:
                if new_sents:
                    all_paths.append({"roles": new_roles, "sentences": new_sents})
                return
            for next_nid, _ in adj[nid]:
                if next_nid not in visited:
                    dfs(next_nid, new_sents, new_roles, visited | {nid}, hop + 1)

        for start_role in start_roles:
            start_nid = chunk_nodes.get(start_role)
            if start_nid:
                dfs(start_nid, [], [], {start_nid}, 0)
        if self.kg["critical_path_sentences"]:
            all_paths.append({
                "roles":     ["PRECEDENT_RELIED", "PROCEDURAL", "REASONING", "ISSUE"],
                "sentences": self.kg["critical_path_sentences"]
            })
        return all_paths

    def rank_paths(self, paths, retrieved_sents):
        if not paths:
            return []
        retrieved_set = {s["sentence"] for s in retrieved_sents}
        scored = []
        for path in paths:
            sents = path["sentences"]; roles = path["roles"]
            if not sents:
                continue
            c_scores = []
            for role in roles:
                nid = self.kg["chunk_nodes"].get(role)
                if nid:
                    n = self.kg["nodes"].get(nid, {})
                    c_scores.append((n.get("entity_centrality", 0.0) +
                                     n.get("path_centrality",   0.0)) / 2)
            centrality = np.mean(c_scores) if c_scores else 0.0
            role_cov   = len(set(roles) & set(KG_CRITICAL_ROLES)) / len(KG_CRITICAL_ROLES)
            if len(sents) > 1:
                embs      = embed_texts(sents[:6])
                sim_mat   = cosine_similarity(embs, embs)
                n         = len(embs)
                coherence = float((sim_mat.sum() - np.trace(sim_mat)) / max(n*(n-1), 1))
            else:
                coherence = 0.5
            overlap_bonus = sum(1 for s in sents if s in retrieved_set) / max(len(sents), 1) * 0.1
            score = (PATH_RANKING_ALPHA * centrality +
                     PATH_RANKING_BETA  * role_cov   +
                     PATH_RANKING_GAMMA * coherence   +
                     overlap_bonus)
            scored.append({"path": path, "score": round(score, 4),
                           "centrality": round(centrality, 4),
                           "role_coverage": round(role_cov, 4),
                           "coherence": round(coherence, 4)})
        scored.sort(key=lambda x: x["score"], reverse=True)
        return scored

    def retrieve(self, doc_text):
        retrieved_sents = self.vector_retrieval(
            "legal reasoning precedent statute decision", top_k=RETRIEVAL_TOP_K)
        paths        = self.graph_traversal(
            start_roles=["PRECEDENT_RELIED", "ISSUE"], depth=GRAPH_TRAVERSAL_DEPTH)
        ranked_paths = self.rank_paths(paths, retrieved_sents)
        return {"retrieved_sentences": retrieved_sents,
                "all_paths":           ranked_paths,
                "top_k_paths":         ranked_paths[:FINAL_TOP_K_PATHS]}

# =========================================================
# STAGE 5: PATH-AWARE SUMMARIZATION
# =========================================================

def serialize_reasoning_paths(top_k_paths, doc_annotation, model_name):
    path_sents_by_role = defaultdict(list)
    for rp in top_k_paths:
        for role, sent in zip(rp["path"]["roles"], rp["path"]["sentences"]):
            tag = STRUCT_TAGS.get(role)
            if tag:
                path_sents_by_role[tag].append(sent)
    for s in doc_annotation["sentences"]:
        tag = STRUCT_TAGS.get(s["role"])
        if tag and s["sentence"] not in path_sents_by_role.get(tag, []):
            path_sents_by_role[tag].append(s["sentence"])
    tag_order = ["[FACTS]", "[PRECEDENT]", "[STATUTE]", "[RATIO]", "[DECISION]"]
    max_per   = max(3, min(10, MODEL_CONFIG[model_name]["max_input"] // (5 * 50)))
    parts     = [f"{tag} {' '.join(path_sents_by_role[tag][:max_per])}"
                 for tag in tag_order if path_sents_by_role.get(tag)]
    struct_str = " ".join(parts)
    max_tokens = MODEL_CONFIG[model_name]["max_input"] // 2
    struct_tok = legal_tokenizer.tokenize(struct_str)[:max_tokens]
    return legal_tokenizer.convert_tokens_to_string(struct_tok)


def stage5_path_aware_summarization(retrieval_output, doc_annotation,
                                    filtered_text, model_name):
    serialized = serialize_reasoning_paths(
        retrieval_output["top_k_paths"], doc_annotation, model_name)
    return generate_summary(model_name, f"{serialized} {filtered_text}")

# =========================================================
# STAGE 6: RL DECOMPOSED FEEDBACK
# =========================================================

class DecomposedRewardComputer:
    def __init__(self, doc_annotation, source_kg, normalized_weights):
        self.doc_annotation     = doc_annotation
        self.source_kg          = source_kg
        self.normalized_weights = normalized_weights
        self.src_sentences      = [s["sentence"] for s in doc_annotation["sentences"]]
        self.src_roles          = [s["role"]     for s in doc_annotation["sentences"]]
        self.src_embs           = (embed_texts(self.src_sentences)
                                   if self.src_sentences else np.zeros((0, 768)))
        self._verified_entity_tokens = set()
        for cat, ents in source_kg["entities"].items():
            for e in ents:
                self._verified_entity_tokens.update(
                    w.lower() for w in e.split() if len(w) > 3)

    def factual_faithfulness(self, summary):
        sum_sents = sent_tokenize(summary)
        if not sum_sents or len(self.src_sentences) == 0:
            return 0.5
        sim_mat = cosine_similarity(embed_texts(sum_sents), self.src_embs)
        return float(sim_mat.max(axis=1).mean())

    def citation_correctness(self, summary):
        sum_entities = extract_entities_spacy(summary)
        summary_tokens = set()
        for cat, ents in sum_entities.items():
            if cat in _LEGAL_CRITICAL_CATEGORIES:
                for e in ents:
                    summary_tokens.update(w.lower() for w in e.split() if len(w) > 3)
        if not summary_tokens:
            return 0.5
        correct = sum(1 for t in summary_tokens if t in self._verified_entity_tokens)
        return correct / len(summary_tokens)

    def reasoning_completeness(self, summary):
        critical_sents = self.source_kg.get("critical_path_sentences", [])
        if not critical_sents:
            return 0.5
        sum_sents = sent_tokenize(summary)
        if not sum_sents:
            return 0.0
        sim_mat = cosine_similarity(embed_texts(critical_sents), embed_texts(sum_sents))
        covered = sum(1 for i in range(len(critical_sents)) if sim_mat[i].max() >= 0.65)
        return covered / len(critical_sents)

    def rhetorical_balance(self, summary):
        sum_sents = sent_tokenize(summary)
        if not sum_sents:
            return 0.0
        role_dist  = Counter(classify_roles(sum_sents))
        total      = len(sum_sents)
        divergence = sum(abs(self.normalized_weights.get(r, 0.0) -
                             role_dist.get(r, 0) / total) for r in LABELS_8)
        return max(0.0, 1.0 - divergence)

    def abstraction_quality(self, summary):
        sum_words = set(word_tokenize(summary.lower()))
        src_words = set(word_tokenize(" ".join(self.src_sentences).lower()))
        if not sum_words:
            return 0.5
        overlap = len(sum_words & src_words) / len(sum_words)
        return max(0.0, min(1.0, 1.0 - abs(overlap - 0.55) * 2))

    def compute_all_rewards(self, summary):
        return {
            "factual_faithfulness":   self.factual_faithfulness(summary),
            "citation_correctness":   self.citation_correctness(summary),
            "reasoning_completeness": self.reasoning_completeness(summary),
            "rhetorical_balance":     self.rhetorical_balance(summary),
            "abstraction_quality":    self.abstraction_quality(summary)
        }


class PPOStyleReranker:
    def __init__(self, reward_computer, case_type="civil"):
        self.reward_computer = reward_computer
        self.weights = RL_CASE_TYPE_WEIGHTS.get(case_type, RL_REWARD_WEIGHTS_DEFAULT)

    def aggregated_reward(self, summary):
        rewards = self.reward_computer.compute_all_rewards(summary)
        r_total = sum(self.weights[k] * rewards[k] for k in rewards)
        return round(r_total, 4), rewards

    def ppo_clip_score(self, reward, baseline=0.5, epsilon=RL_CLIP_EPSILON):
        advantage = reward - baseline
        ratio     = reward / max(baseline, 1e-9)
        clipped   = max(1 - epsilon, min(1 + epsilon, ratio))
        return min(ratio * advantage, clipped * advantage)

    def dpo_preference_score(self, r_chosen, r_rejected, kl_penalty=RL_KL_PENALTY_COEF):
        diff = r_chosen - r_rejected
        return float(np.log(1 / (1 + np.exp(-(diff / max(kl_penalty, 1e-9))))))

    def select_best(self, candidate_summaries):
        evaluated = []
        for name, summary in candidate_summaries.items():
            if not summary.strip():
                continue
            r_total, rewards = self.aggregated_reward(summary)
            ppo_score        = self.ppo_clip_score(r_total)
            evaluated.append({"name": name, "summary": summary,
                               "rewards": rewards, "r_total": r_total,
                               "ppo_score": ppo_score})
        evaluated.sort(key=lambda x: x["ppo_score"], reverse=True)
        dpo_pref = (self.dpo_preference_score(
                        evaluated[0]["r_total"], evaluated[1]["r_total"])
                    if len(evaluated) >= 2 else 0.0)
        best_summary = evaluated[0]["summary"] if evaluated else ""
        return best_summary, evaluated, dpo_pref


def stage6_rl_decomposed_feedback(candidate_summaries, doc_annotation,
                                   source_kg, normalized_weights, doc_text=""):
    print("    [Stage 6] RL Decomposed Feedback...")
    case_type = detect_case_type(doc_text)
    weights   = RL_CASE_TYPE_WEIGHTS.get(case_type, RL_REWARD_WEIGHTS_DEFAULT)
    print(f"      Case type: {case_type} | Weights: {weights}")
    reranker = PPOStyleReranker(
        DecomposedRewardComputer(doc_annotation, source_kg, normalized_weights),
        case_type=case_type)
    best_summary, evaluations, dpo_pref = reranker.select_best(candidate_summaries)
    log = {
        "case_type":            case_type,
        "ner_backend":          _NLP_SOURCE,
        "dynamic_weights":      weights,
        "evaluations":          [{k: v for k, v in e.items() if k != "summary"}
                                 for e in evaluations],
        "dpo_preference_score": round(dpo_pref, 4),
        "selected":             evaluations[0]["name"] if evaluations else "none"
    }
    if evaluations:
        best = evaluations[0]
        print(f"      Best: {best['name']}  R={best['r_total']:.4f}  "
              f"PPO={best['ppo_score']:.4f}")
        for k, v in best["rewards"].items():
            print(f"        {k:<28}: {v:.4f}")
    return best_summary, log

# =========================================================
# STAGE 7: EVALUATION
# =========================================================

def stage7_evaluation(all_predictions, references):
    print("\n" + "="*70)
    print("STAGE 7: EVALUATION (ONLY HERE USE REFERENCE)")
    print("="*70)
    results = {}
    for name, preds in all_predictions.items():
        if not preds:
            continue
        print(f"\n  Evaluating: {name}...")
        metrics = compute_metrics(preds, references)
        results[name] = metrics
        flag = " 🎯" if metrics['rougeL'] >= 0.34 else ""
        print(f"  R1:{metrics['rouge1']:.4f}  R2:{metrics['rouge2']:.4f}  "
              f"RL:{metrics['rougeL']:.4f}{flag}  BS:{metrics['bertscore_f1']:.4f}")

    print("\n" + "="*70)
    print(f"{'Approach':<30} {'ROUGE-1':<10} {'ROUGE-2':<10} {'ROUGE-L':<12} BERTScore")
    print("-"*75)
    for name, m in results.items():
        rl   = m['rougeL']
        flag = " ≥0.34✓" if rl >= 0.34 else f" ({0.34-rl:.4f} short)"
        print(f"{name:<30} {m['rouge1']:<10.4f} {m['rouge2']:<10.4f} "
              f"{rl:.4f}{flag:<12} {m['bertscore_f1']:.4f}")
    best_rl = max(results.items(), key=lambda x: x[1]['rougeL'])
    best_r2 = max(results.items(), key=lambda x: x[1]['rouge2'])
    best_bs = max(results.items(), key=lambda x: x[1]['bertscore_f1'])
    print("\n" + "="*70)
    print(f"🏆 Best ROUGE-L:   {best_rl[0]} → {best_rl[1]['rougeL']:.4f}")
    print(f"🏆 Best ROUGE-2:   {best_r2[0]} → {best_r2[1]['rouge2']:.4f}")
    print(f"🏆 Best BERTScore: {best_bs[0]} → {best_bs[1]['bertscore_f1']:.4f}")
    print("="*70)
    return results

# =========================================================
# MAIN
# =========================================================

def main():
    print("\n" + "="*70)
    print("🚀 7-STAGE PIPELINE  (v3 — utf8 fix + resume from cache)")
    print(f"   NER backend : {_NLP_SOURCE}")
    print("   Cached files in", OUTPUT_DIR, "will be reused automatically.")
    print("="*70)

    print(f"\n📂 Loading train data from {TRAIN_JSON_PATH}...")
    with open(TRAIN_JSON_PATH, encoding='utf-8') as f:   # utf-8 FIX
        train_data = json.load(f)[:MAX_TRAIN_SAMPLES]
    print(f"   {len(train_data)} training samples")

    print(f"\n📂 Loading test data from {TEST_JSON_PATH}...")
    with open(TEST_JSON_PATH, encoding='utf-8') as f:    # utf-8 FIX
        test_data = json.load(f)
    print(f"   {len(test_data)} test samples")

    # Stage 1 — loads from cache
    _, normalized_weights = stage1_role_learning(train_data)

    # Stage 2 setup — loads from cache
    print("\n" + "="*70)
    print("STAGE 2: HYBRID ROLE + SALIENCE SELECTION (TEST PHASE)")
    print("="*70)
    test_annotations = annotate_test_documents(test_data)

    # Per-document stages 3–6
    all_preds = {
        "BART": [], "LED": [], "PEGASUS": [],
        "path_aware_BART": [], "path_aware_LED": [], "path_aware_PEGASUS": [],
        "rl_selected": [], "final": []
    }
    references  = []
    stage6_logs = []

    print("\n🔄 Running per-document pipeline (Stages 3–6)…")
    for test_ann, test_item in tqdm(
            zip(test_annotations, test_data),
            total=len(test_data), desc="Documents"):

        reference = test_item.get("reference_summary", "")
        references.append(reference)
        judgment  = test_item.get("judgment", "")

        doc_len = test_ann["total_sentences"]
        targets = get_adaptive_targets(doc_len)

        # Stage 2
        filtered_text_led, _ = stage2_hybrid_selection(
            test_ann, normalized_weights, targets["LED"])

        # Stage 3
        source_kg = build_role_conditioned_kg(filtered_text_led, test_ann)

        # Stage 4
        retriever        = HybridRetriever(source_kg, test_ann, normalized_weights)
        retrieval_output = retriever.retrieve(filtered_text_led)
        print(f"    [Stage 4] Vector: {len(retrieval_output['retrieved_sentences'])} sents | "
              f"Paths: {len(retrieval_output['all_paths'])} | "
              f"Top-K: {len(retrieval_output['top_k_paths'])}")

        # Stage 5
        stage5_summaries = {}
        for model_name in ["BART", "LED", "PEGASUS"]:
            ft_model, _ = stage2_hybrid_selection(
                test_ann, normalized_weights, targets[model_name])
            direct_sum = generate_summary(model_name, ft_model)
            all_preds[model_name].append(direct_sum)
            path_sum = stage5_path_aware_summarization(
                retrieval_output, test_ann, ft_model, model_name)
            all_preds[f"path_aware_{model_name}"].append(path_sum)
            stage5_summaries[model_name]           = direct_sum
            stage5_summaries[f"path_{model_name}"] = path_sum

        # Stage 6
        rl_best, rl_log = stage6_rl_decomposed_feedback(
            stage5_summaries, test_ann, source_kg, normalized_weights, judgment)
        all_preds["rl_selected"].append(rl_best)
        all_preds["final"].append(rl_best)

        if len(stage6_logs) < 5:
            stage6_logs.append({
                "doc_id":   test_ann["doc_id"],
                "log":      rl_log,
                "kg_metrics": source_kg["graph_metrics"],
                "retrieval_stats": {
                    "vector_retrieved": len(retrieval_output["retrieved_sentences"]),
                    "total_paths":      len(retrieval_output["all_paths"]),
                    "top_k_paths":      len(retrieval_output["top_k_paths"])
                }
            })

    # ── Save stage 6 logs ─────────────────────────────────
    logs_path = os.path.join(OUTPUT_DIR, "stage6_rl_logs.json")
    with open(logs_path, 'w', encoding='utf-8') as f:    # utf-8 FIX
        json.dump(stage6_logs, f, indent=2, ensure_ascii=False)
    print(f"\n✅ Stage 6 logs → {logs_path}")

    # ── Save all summaries ────────────────────────────────
    print("\n💾 Saving all summaries (utf-8)…")
    for name, preds in all_preds.items():
        if not preds:
            continue
        out = [{"id":                item.get("id", idx),
                "generated_summary": pred,
                "reference_summary": references[idx]}
               for idx, (item, pred) in enumerate(zip(test_data, preds))]
        out_path = os.path.join(OUTPUT_DIR, f"{name}_summaries.json")
        with open(out_path, 'w', encoding='utf-8') as f:  # utf-8 FIX ← THE KEY LINE
            json.dump(out, f, indent=2, ensure_ascii=False)
        print(f"  ✅ {name:25s} → {out_path}")

    # ── Stage 7: Evaluation ───────────────────────────────
    eval_preds = {k: v for k, v in all_preds.items()
                  if v and len(v) == len(references)}
    results = stage7_evaluation(eval_preds, references)

    # ── Save complete results ─────────────────────────────
    complete = {
        "pipeline":    "7-Stage Ensemble Legal Summarization Pipeline v3 (utf8 fix)",
        "ner_backend": _NLP_SOURCE,
        "stages": {
            "1": "Role Learning — cached",
            "2": "Hybrid Role + Salience — cached annotations",
            "3": "Role-Conditioned KG — spaCy NER + dep-parse",
            "4": "Hybrid Retrieval — Vector + Graph Traversal + Path Ranking",
            "5": "Path-Aware Abstractive Summarization",
            "6": "RL Decomposed Feedback — PPO/DPO",
            "7": "Evaluation — ROUGE-1/2/L, BERTScore-F1"
        },
        "test_samples": len(test_data),
        "results":      results
    }
    results_path = os.path.join(OUTPUT_DIR, "complete_results_v3.json")
    with open(results_path, 'w', encoding='utf-8') as f:  # utf-8 FIX
        json.dump(complete, f, indent=2, ensure_ascii=False)

    print(f"\n✅ Complete results → {results_path}")
    print(f"✅ All outputs     → {OUTPUT_DIR}/")
    print("\n" + "="*70)
    print("✅ 7-STAGE PIPELINE v3 (utf8 fix) COMPLETE!")
    print("="*70)


if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\n⚠️  Interrupted by user")
    except Exception as e:
        import traceback
        print(f"\n❌ Error: {e}")
        traceback.print_exc()

📥 Downloading NLTK data...
🚀 Device: cuda

📚 Loading spaCy Legal NER model...


/home/RSlab/.conda/envs/legal-nlp/lib/python3.10/site-packages/spacy/util.py:969: UserWarning: [W095] Model 'en_legal_ner_trf' (3.2.0) was trained with spaCy v3.2.2 and may not be 100% compatible with the current version (3.8.11). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)
/home/RSlab/.conda/envs/legal-nlp/lib/python3.10/site-packages/spacy_transformers/layers/hf_shim.py:137: UserWarning: Error loading saved torch state_dict with strict=True, likely due to differences between 'transformers' versions. Attempting to load with strict=False as a fallback...

If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current 'transformers' and 'spacy-transformers' versions. For more details and available updates, run: python -m spacy validate
  war

  ℹ️  Added 'sentencizer' to 'en_legal_ner_trf' pipeline
  ✅ Loaded spaCy model: en_legal_ner_trf  (pipes: ['transformer', 'ner', 'sentencizer'])

7-STAGE PIPELINE  (v3 — utf8 fix)
  NER backend: en_legal_ner_trf
  OUTPUT_DIR: ./pipeline_7stage_v3_outputs

📚 Loading HSLN role classifier...
✅ HSLN loaded

📚 Loading InLegalBERT...


/home/RSlab/.conda/envs/legal-nlp/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✅ InLegalBERT loaded

📚 Loading summarization models...
  Loading BART...
  ✅ BART loaded
  Loading LED...
  ✅ LED loaded
  Loading PEGASUS...
  ✅ PEGASUS loaded

📊 Loading evaluation metrics...
✅ Metrics loaded


🚀 7-STAGE PIPELINE  (v3 — utf8 fix + resume from cache)
   NER backend : en_legal_ner_trf
   Cached files in ./pipeline_7stage_v3_outputs will be reused automatically.

📂 Loading train data from train.json...
   3000 training samples

📂 Loading test data from test.json...
   100 test samples

STAGE 1: ROLE LEARNING (TRAINING PHASE)
📂 Loading cached train annotations from ./pipeline_7stage_v3_outputs/train_annotations_8label.json
📂 Loading cached role contribution scores from ./pipeline_7stage_v3_outputs/role_contribution_scores.json
📂 Loading cached normalized weights from ./pipeline_7stage_v3_outputs/normalized_role_weights.json

  📊 Learned Role Importance Weights:
  ------------------------------------------------------------
  PRECEDENT_RELIED         : 0.2465  ██████████

...

      Best: path_BART  R=0.5816  PPO=0.0949
        factual_faithfulness        : 0.9535
        citation_correctness        : 1.0000
        reasoning_completeness      : 0.5000
        rhetorical_balance          : 0.0000
        abstraction_quality         : 0.1820
    [Stage 3] Building KG  (NER: en_legal_ner_trf)
    [Stage 3] Entities: {'dates': 18, 'courts': 4, 'cases': 6, 'locations': 2, 'persons': 3, 'provisions': 16, 'statutes': 6, 'judges': 1, 'organizations': 4}
    [Stage 3] Relations: 0
    [Stage 3] KG: 44 nodes, 35 edges, coverage=0.88, depth=5
    [Stage 4] Vector: 10 sents | Paths: 3 | Top-K: 3
    [Stage 6] RL Decomposed Feedback...
      Case type: constitutional | Weights: {'factual_faithfulness': 0.25, 'citation_correctness': 0.15, 'reasoning_completeness': 0.35, 'rhetorical_balance': 0.15, 'abstraction_quality': 0.1}


Documents: 100%|████████████████████████████████████████████████████████████████████| 100/100 [2:38:11<00:00, 94.91s/it]

      Best: path_LED  R=0.7592  PPO=0.3110
        factual_faithfulness        : 0.9509
        citation_correctness        : 1.0000
        reasoning_completeness      : 1.0000
        rhetorical_balance          : 0.0000
        abstraction_quality         : 0.2143

✅ Stage 6 logs → ./pipeline_7stage_v3_outputs/stage6_rl_logs.json

💾 Saving all summaries (utf-8)…
  ✅ BART                      → ./pipeline_7stage_v3_outputs/BART_summaries.json
  ✅ LED                       → ./pipeline_7stage_v3_outputs/LED_summaries.json
  ✅ PEGASUS                   → ./pipeline_7stage_v3_outputs/PEGASUS_summaries.json
  ✅ path_aware_BART           → ./pipeline_7stage_v3_outputs/path_aware_BART_summaries.json
  ✅ path_aware_LED            → ./pipeline_7stage_v3_outputs/path_aware_LED_summaries.json
  ✅ path_aware_PEGASUS        → ./pipeline_7stage_v3_outputs/path_aware_PEGASUS_summaries.json
  ✅ rl_selected               → ./pipeline_7stage_v3_outputs/rl_selected_summaries.json
  ✅ final            


/home/RSlab/.conda/envs/legal-nlp/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  R1:0.3687  R2:0.1881  RL:0.2106  BS:0.8497

  Evaluating: LED...
  R1:0.5001  R2:0.2654  RL:0.2761  BS:0.8529

  Evaluating: PEGASUS...
  R1:0.3805  R2:0.1901  RL:0.2151  BS:0.8431

  Evaluating: path_aware_BART...
  R1:0.3543  R2:0.1753  RL:0.2043  BS:0.8475

  Evaluating: path_aware_LED...
  R1:0.4980  R2:0.2636  RL:0.2677  BS:0.8527

  Evaluating: path_aware_PEGASUS...
  R1:0.3713  R2:0.1823  RL:0.2081  BS:0.8425

  Evaluating: rl_selected...
  R1:0.4390  R2:0.2329  RL:0.2467  BS:0.8486

  Evaluating: final...
  R1:0.4390  R2:0.2329  RL:0.2467  BS:0.8486

Approach                       ROUGE-1    ROUGE-2    ROUGE-L      BERTScore
---------------------------------------------------------------------------
BART                           0.3687     0.1881     0.2106 (0.1294 short) 0.8497
LED                            0.5001     0.2654     0.2761 (0.0639 short) 0.8529
PEGASUS                        0.3805     0.1901     0.2151 (0.1249 short) 0.8431
path_aware_BART                0.35